# Extended optical-flow torus

This notebook reproduces the extended-torus experiment using the current
`circle_bundles` API. It replaces the recovered migration source without
modifying that preserved file.

## Execution profiles

- **quick** samples 400 patches per Sintel frame and selects 25,000 points from
  $X(300,50)$. It is intended for clean-environment validation.
- **paper** samples 4,000 patches per frame and selects the manuscript-scale
  125,000 points from $X(1500,50)$.

Set `OPTICAL_FLOW_PROFILE` to a profile TOML file. Set
`OPTICAL_FLOW_PREPROCESSED` to a portable `.npz` artifact to bypass sampling,
or set `MPI_SINTEL_FLOW_DIR` to the extracted `training/flow` directory.

In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from persim import plot_diagrams
from ripser import ripser

import circle_bundles as cb
from optical_flow_experiments import (
    fit_extended_torus,
    load_experiment_config,
    load_preprocessed_npz,
    sample_and_preprocess,
    save_preprocessed_npz,
    select_dense_core,
)

ROOT = Path.cwd().resolve()
if not (ROOT / "configs").is_dir() and (ROOT.parent / "configs").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "configs").is_dir():
    raise RuntimeError("Could not locate the repository root and its configs directory.")

profile_path = Path(os.environ.get("OPTICAL_FLOW_PROFILE", ROOT / "configs/quick.toml"))
config = load_experiment_config(profile_path)
print(f"Profile: {config.name}")
print(config.description)
print(f"Canonical random seed: {config.random_seed} (historical seed: {config.historical_seed})")

## Prepare or load the high-contrast sample

In [ ]:
artifact_override = os.environ.get("OPTICAL_FLOW_PREPROCESSED")
artifact_path = (
    Path(artifact_override).expanduser()
    if artifact_override
    else ROOT / "data" / f"preprocessed_{config.name}.v1.npz"
)

if artifact_path.is_file():
    patch_table = load_preprocessed_npz(artifact_path)
    print(f"Loaded {len(patch_table):,} preprocessed patches from {artifact_path}")
else:
    sintel_flow_dir = os.environ.get("MPI_SINTEL_FLOW_DIR")
    if not sintel_flow_dir:
        raise RuntimeError(
            "No preprocessed artifact was found. Set MPI_SINTEL_FLOW_DIR to "
            "MPI-Sintel-complete/training/flow or set OPTICAL_FLOW_PREPROCESSED."
        )
    patch_table = sample_and_preprocess(sintel_flow_dir, config)
    save_preprocessed_npz(patch_table, artifact_path)
    print(f"Saved {len(patch_table):,} preprocessed patches to {artifact_path}")

## Select the configured dense core

In [ ]:
core = select_dense_core(
    patch_table,
    density_k=config.torus.density_k,
    density_fraction=config.torus.density_fraction,
)
data = core.data
assert len(data) == config.torus.expected_patch_count, (
    f"Expected {config.torus.expected_patch_count:,} points, observed {len(data):,}."
)
print(
    f"X({core.density_k}, {int(100 * core.density_fraction)}) contains "
    f"{len(data):,} normalized 3 x 3 flow patches."
)

In [ ]:
patch_vis = cb.make_patch_visualizer()
fig = cb.show_data_vis(
    data,
    patch_vis,
    sampling_method=None,
    max_samples=30,
)
plt.show()

## Direct persistent homology

A direct computation does not exhibit the two long one-dimensional classes and
long two-dimensional class expected from a clean torus.

In [ ]:
diagrams = ripser(data, maxdim=2, n_perm=500)["dgms"]
plot_diagrams(diagrams, show=True)

## Bundle analysis over predominant direction

The feature map sends each flow patch to its predominant unoriented axis in
$\mathbb{RP}^1$. Local circular coordinates are synchronized into a global
fiber angle after checking the characteristic classes.

In [ ]:
fit = fit_extended_torus(
    data,
    n_landmarks=config.torus.cover_landmarks,
    overlap=config.torus.cover_overlap,
    show_summaries=True,
)
predominant_directions = fit.predominant_directions
directionalities = fit.directionalities
fiber_angles = fit.fiber_angles
print(f"Global fiber coordinates computed for {len(fiber_angles):,} patches.")

In [ ]:
to_view = [3, 6, 14]
fig, axes = cb.get_local_pca(
    data=data,
    U=fit.cover.U,
    show=True,
    to_view=to_view,
)

In [ ]:
n_samples = 8
labels = [
    fr"$\theta = {np.round(direction / np.pi, 2)}\pi$"
    for direction in predominant_directions
]
fig = cb.show_data_vis(
    data,
    patch_vis,
    label_func=labels,
    angles=predominant_directions,
    sampling_method="angle",
    max_samples=n_samples,
)
plt.show()

## High- and low-directionality coordinates

In [ ]:
high_mask = directionalities > 0.8
high_coordinates = np.column_stack([
    predominant_directions[high_mask],
    fiber_angles[high_mask],
])
print(f"{high_mask.sum():,} high-directionality patches")
fig, ax, selected = cb.scatter_lattice_vis(
    data[high_mask],
    high_coordinates,
    patch_vis,
    per_row=5,
    per_col=9,
    padding=0,
)
plt.show()

In [ ]:
low_mask = directionalities < 0.7
low_coordinates = np.column_stack([
    predominant_directions[low_mask],
    fiber_angles[low_mask],
])
print(f"{low_mask.sum():,} low-directionality patches")
fig, ax, selected = cb.scatter_lattice_vis(
    data[low_mask],
    low_coordinates,
    patch_vis,
    per_row=5,
    per_col=9,
    padding=0,
)
plt.show()

## Record run metrics

In [ ]:
def top_finite_lifetimes(diagram, count=5):
    lifetimes = diagram[:, 1] - diagram[:, 0]
    finite = lifetimes[np.isfinite(lifetimes)]
    return [float(value) for value in np.sort(finite)[::-1][:count]]

metrics = {
    "profile": config.name,
    "random_seed": config.random_seed,
    "preprocessed_patch_count": len(patch_table),
    "dense_core_patch_count": len(data),
    "density_k": core.density_k,
    "density_fraction": core.density_fraction,
    "cover_landmarks": config.torus.cover_landmarks,
    "cover_overlap": config.torus.cover_overlap,
    "high_directionality_count": int(high_mask.sum()),
    "low_directionality_count": int(low_mask.sum()),
    "class_summary": fit.classes.summary_text,
    "top_persistence_lifetimes": {
        f"H{dimension}": top_finite_lifetimes(diagram)
        for dimension, diagram in enumerate(diagrams)
    },
}
metrics_path = ROOT / "results" / config.name / "extended_torus_metrics.json"
metrics_path.parent.mkdir(parents=True, exist_ok=True)
metrics_path.write_text(json.dumps(metrics, indent=2, sort_keys=True) + "\n")
print(f"Wrote run metrics to {metrics_path}")
metrics